# RAW → BRONZE: Ingestion de Customers

Este notebook ejecuta el script de ingesta que:
1. Se conecta a MySQL via JDBC usando PySpark
2. Lee la tabla `customers`
3. Guarda los datos localmente en formato Parquet (capa Bronze)

In [ ]:
import sys
from pathlib import Path

# Agrega el directorio raiz del proyecto al path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

## 1. Verificar configuracion de base de datos

In [ ]:
from src.utils.config_loader import load_db_config, get_jdbc_url

db_config = load_db_config()
jdbc_url = get_jdbc_url(db_config)

print(f"Host     : {db_config['host']}")
print(f"Port     : {db_config['port']}")
print(f"Database : {db_config['database']}")
print(f"User     : {db_config['user']}")
print(f"JDBC URL : {jdbc_url}")

## 2. Ejecutar el script RAW → BRONZE

In [ ]:
from src.raw_to_bronze.customers_ingestion import ingest_customers

total_records = ingest_customers()
print(f'\nIngestion completada: {total_records} registros guardados en Bronze.')

## 3. Validar la capa Bronze

In [ ]:
from src.utils.spark_session import create_spark_session

BRONZE_PATH = PROJECT_ROOT / 'data' / 'bronze' / 'customers'

spark = create_spark_session('notebook_validation')
df_bronze = spark.read.parquet(str(BRONZE_PATH))

print(f'Schema Bronze:')
df_bronze.printSchema()

In [ ]:
print(f'Total registros: {df_bronze.count()}')
df_bronze.show(10, truncate=False)

In [ ]:
# Estadisticas basicas
df_bronze.describe().show()

In [ ]:
spark.stop()
print('SparkSession cerrada.')